# 🎭 Emotion Recognition using Ensemble Learning

**Optimized version** - Faster execution with fewer cross-validation folds

Based on the paper:
**"Gradient Boosting, AdaBoost, and LightGBM for Emotion Recognition: A Comparative Analysis on Wearable Sensor Data"** (ICoICT 2024)

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/winwin52/emotion-recognition-smartwatch.git
%cd emotion-recognition-smartwatch
!pip install lightgbm pyyaml -q
import warnings
warnings.filterwarnings('ignore')  # Suppress LightGBM warnings

In [ ]:
import os
import glob
import numpy as np
import time

from sklearn import metrics
from sklearn import preprocessing
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

# Get feature files
features_dir = 'features'
mo_files = sorted(glob.glob(os.path.join(features_dir, 'features_mo_*')))
mu_files = sorted(glob.glob(os.path.join(features_dir, 'features_mu_*')))
mw_files = sorted(glob.glob(os.path.join(features_dir, 'features_mw_*')))

print(f'📁 Movie (mo): {len(mo_files)} files')
print(f'📁 Music (mu): {len(mu_files)} files')
print(f'📁 Music+Walk (mw): {len(mw_files)} files')

In [ ]:
def run_experiment(fnames, condition_name, n_estimators=50, n_splits=5, include_neutral=False):
    """
    Optimized experiment with fewer folds for faster execution.
    """
    print(f'\n{"="*60}')
    print(f'📊 {condition_name} - Binary Classification (Happy vs Sad)')
    print(f'{"="*60}')

    # Aggregate all data
    all_data = []
    for fname in fnames:
        data = np.loadtxt(fname, delimiter=',')
        if not include_neutral:
            data = np.delete(data, np.where(data[:, -1] == 0), axis=0)
        all_data.append(data)
    
    data = np.vstack(all_data)
    np.random.shuffle(data)
    
    X, y = data[:, :-1], data[:, -1]
    X = preprocessing.scale(X)
    
    print(f'   Total samples: {len(y)}, Labels: {np.unique(y, return_counts=True)}')

    # Define models
    models = {
        'Baseline': DummyClassifier(strategy='most_frequent'),
        'Logistic Reg': LogisticRegression(max_iter=500, random_state=SEED),
        'Random Forest': RandomForestClassifier(n_estimators=n_estimators, random_state=SEED, n_jobs=-1),
        'AdaBoost': AdaBoostClassifier(n_estimators=n_estimators, random_state=SEED),
        'Gradient Boost': GradientBoostingClassifier(n_estimators=n_estimators, random_state=SEED),
        'LightGBM': lgb.LGBMClassifier(n_estimators=n_estimators, random_state=SEED, verbose=-1, force_row_wise=True)
    }

    # Cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    results = {}
    for name, clf in models.items():
        start = time.time()
        acc_list, f1_list, auc_list = [], [], []
        
        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            
            acc = metrics.accuracy_score(y_test, y_pred)
            f1 = metrics.f1_score(y_test, y_pred, average='binary' if len(np.unique(y_test)) == 2 else 'weighted')
            
            try:
                y_proba = clf.predict_proba(X_test)
                auc = metrics.roc_auc_score(y_test, y_proba[:, 1]) if len(np.unique(y_test)) > 1 else 0.5
            except:
                auc = 0.5
            
            acc_list.append(acc)
            f1_list.append(f1)
            auc_list.append(auc)
        
        elapsed = time.time() - start
        results[name] = {
            'acc': np.mean(acc_list),
            'f1': np.mean(f1_list),
            'auc': np.mean(auc_list)
        }
        print(f'   {name:15s}: Acc={results[name]["acc"]:.4f}, F1={results[name]["f1"]:.4f}, AUC={results[name]["auc"]:.4f} ({elapsed:.1f}s)')
    
    return results

In [ ]:
# 🏃 Run experiments for all conditions
print('\n' + '🎬'*20 + ' BINARY CLASSIFICATION ' + '🎬'*20)

results_mo = run_experiment(mo_files, 'Movie (mo)', n_estimators=50, n_splits=5)
results_mu = run_experiment(mu_files, 'Music (mu)', n_estimators=50, n_splits=5)
results_mw = run_experiment(mw_files, 'Music+Walk (mw)', n_estimators=50, n_splits=5)

In [ ]:
# 📊 Summary Table
print('\n' + '='*80)
print('📊 RESULTS SUMMARY - Binary Classification (Happy vs Sad)')
print('='*80)
print(f'\n{"Model":<18} {"Movie (mo)":>15} {"Music (mu)":>15} {"Music+Walk (mw)":>15}')
print('-'*65)

for name in ['Baseline', 'Logistic Reg', 'Random Forest', 'AdaBoost', 'Gradient Boost', 'LightGBM']:
    mo_acc = results_mo[name]['acc']
    mu_acc = results_mu[name]['acc']
    mw_acc = results_mw[name]['acc']
    print(f'{name:<18} {mo_acc:>14.2%} {mu_acc:>14.2%} {mw_acc:>14.2%}')

print('\n' + '='*80)
print('🏆 BEST MODEL PER CONDITION:')
print(f'   Movie (mo):       {max(results_mo, key=lambda k: results_mo[k]["acc"])} ({max(results_mo.values(), key=lambda v: v["acc"])["acc"]:.2%})')
print(f'   Music (mu):       {max(results_mu, key=lambda k: results_mu[k]["acc"])} ({max(results_mu.values(), key=lambda v: v["acc"])["acc"]:.2%})')
print(f'   Music+Walk (mw):  {max(results_mw, key=lambda k: results_mw[k]["acc"])} ({max(results_mw.values(), key=lambda v: v["acc"])["acc"]:.2%})')

In [ ]:
# 🎯 Run Multi-class Classification (Happy, Neutral, Sad)
print('\n\n' + '🎭'*20 + ' MULTI-CLASS CLASSIFICATION ' + '🎭'*20)

results_mo_3 = run_experiment(mo_files, 'Movie (mo) - 3 class', n_estimators=50, n_splits=5, include_neutral=True)
results_mu_3 = run_experiment(mu_files, 'Music (mu) - 3 class', n_estimators=50, n_splits=5, include_neutral=True)
results_mw_3 = run_experiment(mw_files, 'Music+Walk (mw) - 3 class', n_estimators=50, n_splits=5, include_neutral=True)

In [ ]:
# 📊 Multi-class Summary
print('\n' + '='*80)
print('📊 RESULTS SUMMARY - Multi-class Classification (Happy, Neutral, Sad)')
print('='*80)
print(f'\n{"Model":<18} {"Movie (mo)":>15} {"Music (mu)":>15} {"Music+Walk (mw)":>15}')
print('-'*65)

for name in ['Baseline', 'Logistic Reg', 'Random Forest', 'AdaBoost', 'Gradient Boost', 'LightGBM']:
    mo_acc = results_mo_3[name]['acc']
    mu_acc = results_mu_3[name]['acc']
    mw_acc = results_mw_3[name]['acc']
    print(f'{name:<18} {mo_acc:>14.2%} {mu_acc:>14.2%} {mw_acc:>14.2%}')

print('\n' + '='*80)
print('🏆 BEST MODEL PER CONDITION (Multi-class):')
print(f'   Movie (mo):       {max(results_mo_3, key=lambda k: results_mo_3[k]["acc"])} ({max(results_mo_3.values(), key=lambda v: v["acc"])["acc"]:.2%})')
print(f'   Music (mu):       {max(results_mu_3, key=lambda k: results_mu_3[k]["acc"])} ({max(results_mu_3.values(), key=lambda v: v["acc"])["acc"]:.2%})')
print(f'   Music+Walk (mw):  {max(results_mw_3, key=lambda k: results_mw_3[k]["acc"])} ({max(results_mw_3.values(), key=lambda v: v["acc"])["acc"]:.2%})')

---
## 📝 Notes

Based on the original paper (ICoICT 2024):
| Condition | Best Model | Reported Accuracy |
|-----------|------------|-------------------|
| Movie + Walk | LightGBM | 91.8% (binary) |
| Music + Walk | LightGBM | 85.3% (binary) |
| Music while Walk | LightGBM | 91.8% (binary) |

The algorithm replicates:
- **Gradient Boosting**: Sequential tree ensemble with error correction
- **AdaBoost**: Adaptive boosting with weight adjustment
- **LightGBM**: Leaf-wise tree growth with GOSS and EFB optimizations